In [1]:
import pandas as pd

def load_and_convert(csv_file):
    # Load user data
    df = pd.read_csv(csv_file, parse_dates=["timestamp"])

    # Sort by time
    df = df.sort_values("timestamp").reset_index(drop=True)

    # Time-based features
    df["hour"] = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.dayofweek  # Monday=0, Sunday=6
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
    df["minutes_since_midnight"] = df["timestamp"].dt.hour * 60 + df["timestamp"].dt.minute

    # Track last occurrences of events
    last_drink, last_eat, last_outside = None, None, None
    drink_gaps, eat_gaps, outside_gaps = [], [], []

    for idx, row in df.iterrows():
        current_time = row["timestamp"]

        # Time since last drink
        if row["activity"] == "drink":
            if last_drink is not None:
                drink_gaps.append((current_time - last_drink).total_seconds() / 60)
            last_drink = current_time
        df.loc[idx, "time_since_last_drink"] = (
            (current_time - last_drink).total_seconds() / 60 if last_drink else None
        )

        # Time since last eat
        if row["activity"] == "eat":
            if last_eat is not None:
                eat_gaps.append((current_time - last_eat).total_seconds() / 60)
            last_eat = current_time
        df.loc[idx, "time_since_last_eat"] = (
            (current_time - last_eat).total_seconds() / 60 if last_eat else None
        )

        # Time since last outside
        if row["activity"] == "outside_in":
            last_outside = current_time
        df.loc[idx, "time_since_last_outside"] = (
            (current_time - last_outside).total_seconds() / 60 if last_outside else None
        )

    # Rolling averages
    df["avg_drink_gap"] = sum(drink_gaps) / len(drink_gaps) if drink_gaps else None
    df["avg_eat_gap"] = sum(eat_gaps) / len(eat_gaps) if eat_gaps else None

    # Target variable
    df["minutes_until_next"] = (
        df["timestamp"].shift(-1) - df["timestamp"]
    ).dt.total_seconds() / 60

    return df

if __name__ == "__main__":
    # Example usage
    input_file = "Dataset/Userdata.csv"
    features = load_and_convert(input_file)
    print(features.head(20))
    features.to_csv("Dataset/Features.csv", index=False)

AttributeError: Can only use .dt accessor with datetimelike values

In [2]:
import pandas as pd

# Load data and get the date
df = pd.read_csv("Dataset/Userdata.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.dropna(subset=['timestamp'])

df['date'] = df['timestamp'].dt.date

# Count occurrences per day
daily = df.groupby(['date', 'activity']).size().unstack(fill_value=0).reset_index()

# Add Features (Day context)
daily['date'] = pd.to_datetime(daily['date'])
daily['day_of_week'] = daily['date'].dt.dayofweek
daily['is_weekend'] = daily['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

# CREATE THE TARGET: The count of the NEXT day, shift the counts up by 1 row so "Today" has "Tomorrow's" count as a label
daily['target_drink_tomorrow'] = daily['drink'].shift(-1)
daily['target_eat_tomorrow'] = daily['eat'].shift(-1)

# Remove the last row because we don't know the future yet
daily = daily.dropna()

daily.to_csv("Dataset/Daily_Features.csv", index=False)
print("New Daily Dataset Created!")

New Daily Dataset Created!


In [ ]:
import pandas as pd

# Load data, read everything as strings first to avoid errors
df = pd.read_csv("Dataset/Userdata.csv", dtype={'timestamp': str})

# Convert timestamp allowing MIXED formats
df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed', errors='coerce')

# Drop rows that couldn't be converted (header rows and other data that shouldn't be there)
df = df.dropna(subset=['timestamp'])

# Extract Date
df['date'] = df['timestamp'].dt.date

# Group and Count Daily Activities
daily = df.groupby(['date', 'activity']).size().unstack(fill_value=0).reset_index()

# Add Day Features
daily['date'] = pd.to_datetime(daily['date'])
daily['day_of_week'] = daily['date'].dt.dayofweek
daily['is_weekend'] = daily['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

# Add Weekly Memory (7-day rolling average)
# min_periods=1 ensures we don't lose data if there's a gap
daily['drink_7day_avg'] = daily['drink'].rolling(window=7, min_periods=1).mean()
daily['eat_7day_avg'] = daily['eat'].rolling(window=7, min_periods=1).mean()

# Create Targets (Tomorrow's count)
daily['target_drink_tomorrow'] = daily['drink'].shift(-1)
daily['target_eat_tomorrow'] = daily['eat'].shift(-1)

# Drop the last row (since we can't predict "tomorrow" for the final day)
daily = daily.dropna()

# Save
daily.to_csv("Dataset/Daily_Features.csv", index=False)

print(f"Success! Processed {len(df)} rows.")
print(f"Daily Features file now has {len(daily)} days of data.")
print("First 5 rows:", daily.head())

Success! Processed 2772 rows.
Daily Features file now has 227 days of data.
First 5 rows: activity       date  drink  eat  outside_in  outside_out  day_of_week  \
0        2018-09-01      6    4           0            0            5   
1        2018-09-02      6    4           0            0            6   
2        2018-09-03     12    5           0            0            0   
3        2018-09-04     11    3           0            0            1   
4        2018-09-05      8    3           0            0            2   

activity  is_weekend  drink_7day_avg  eat_7day_avg  target_drink_tomorrow  \
0                  1            6.00      4.000000                    6.0   
1                  1            6.00      4.000000                   12.0   
2                  0            8.00      4.333333                   11.0   
3                  0            8.75      4.000000                    8.0   
4                  0            8.60      3.800000                    9.0   

activity

In [5]:
import pandas as pd
import random

# Number of rows for the test dataset
num_rows = 30

# Initialize lists
days = []
is_weekends = []
drinks = []
eats = []

# Starting state for rolling averages (realistic values)
current_drink_avg = 7.5
current_eat_avg = 3.2

data = []

for i in range(num_rows):
    # Day of week (0-6)
    dow = i % 7
    weekend = 1 if dow >= 5 else 0
    
    # Drink count: slightly higher on weekends
    if weekend:
        drink = random.randint(8, 12)
        eat = random.randint(3, 5)
    else:
        drink = random.randint(5, 9)
        eat = random.randint(3, 4)
    
    # Simulate a shifting rolling average
    # In a real scenario, this would be calculated from the previous 7 days.
    # Here we just keep it in a realistic range.
    drink_avg = round(random.uniform(6.5, 8.5), 2)
    eat_avg = round(random.uniform(3.0, 3.8), 2)
    
    data.append([dow, weekend, drink, eat, drink_avg, eat_avg])

# Create DataFrame
test_df = pd.DataFrame(data, columns=['day_of_week', 'is_weekend', 'drink', 'eat', 'drink_7day_avg', 'eat_7day_avg'])

# Save to CSV
test_df.to_csv("Larger_Test_Dataset.csv", index=False)

print(test_df.head(10))

   day_of_week  is_weekend  drink  eat  drink_7day_avg  eat_7day_avg
0            0           0      6    4            7.39          3.50
1            1           0      8    3            7.14          3.34
2            2           0      6    3            7.31          3.19
3            3           0      8    3            6.53          3.37
4            4           0      9    4            8.25          3.11
5            5           1      9    4            8.03          3.66
6            6           1     12    5            6.78          3.14
7            0           0      7    4            8.20          3.31
8            1           0      7    4            6.72          3.15
9            2           0      7    4            7.17          3.52


In [3]:
#Converting files to android
import joblib
from skl2onnx import to_onnx
from skl2onnx.common.data_types import FloatTensorType

# Load your trained models
model_drink = joblib.load("daily_drink_forecast.roboai")
model_eat = joblib.load("daily_eat_forecast.roboai")

# Define the input shape (The model expects 6 numbers at a time)
# Features: [day_of_week, is_weekend, drink_count, eat_count, drink_avg, eat_avg]
initial_type = [('float_input', FloatTensorType([None, 6]))]

# Convert to ONNX
onnx_drink = to_onnx(model_drink, initial_types=initial_type)
onnx_eat = to_onnx(model_eat, initial_types=initial_type)

# Save for Android
with open("drink_model.onnx", "wb") as f:
    f.write(onnx_drink.SerializeToString())
with open("eat_model.onnx", "wb") as f:
    f.write(onnx_eat.SerializeToString())

print("Conversion complete! Move 'drink_model.onnx' and 'eat_model.onnx' to your Android project.")

Conversion complete! Move 'drink_model.onnx' and 'eat_model.onnx' to your Android project.
